# SDN on ME4OH full-field OFAM3 SST data
MLP built from three linear layers (LSTM removed)

Note: Assumptions as of 30/06/26
- OFAM3 data is stored for Thursdays across the 1979-2014 time period.
- Pending confirmation from the ME4OH team, I have assumed that the data is the full-field SST values generated by the simulator model (OFAM3) averaged across a calendar week

In [ ]:
# Standard imports
from collections import Counter

# Third-party imports
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import mpl_scatter_density  # adds projection='scatter_density'
import numpy as np
import pandas as pd
from matplotlib import cm
from matplotlib.colors import LinearSegmentedColormap, ListedColormap, Normalize
from scipy.interpolate import interpn
from skimage.metrics import structural_similarity as SSI
from sklearn.metrics import mean_absolute_error as MAE
from sklearn.metrics import mean_squared_error as MSE
from sklearn.metrics import r2_score

# Local imports
from Data.load_sst_en411_data import crop_df_to_area
from Data.preprocess_sst_ofam3_shred import NA_DATA_PATH
from helper_functions.evaluation import (
    area_weighted_mean,
    get_lats_longs,
    mask_array_by_lat_long,
)
from helper_functions.plotting_functions import (
    plot_compare_sst_recon,
    plot_sensors,
    plot_sst_map,
)
from helper_functions.saving import unpack_metadata

np.random.seed(42)

In [ ]:
colours = [ "#ffb000", "#dc267f", "#1589e8", "#631ff3", "#fe5100", "#000000"]  # IBM colourblind palette

## Set up

In [ ]:
exp_index = 4
num_sensors = 5
lags = 52

In [ ]:
# Load data
lats, longs = get_lats_longs(NA_DATA_PATH)
sensor_locations, sensor_coords, _, test_lag_dates = unpack_metadata(f"Reconstructions/Static/Exp{exp_index}/metadata_n{num_sensors}.json")

In [ ]:
# Load reconstructions
test_recons = np.load(f"Reconstructions/Static/Exp{exp_index}/recons_n{num_sensors}_l{lags}.npy")
ground_truth = np.load(f"Reconstructions/Static/Exp{exp_index}/truth_n{num_sensors}_l{lags}.npy")

test_recons_shred = np.load(f"Reconstructions/Static/Exp1/recons_n{num_sensors}_l{lags}.npy")
ground_truth_shred = np.load(f"Reconstructions/Static/Exp1/truth_n{num_sensors}_l{lags}.npy")
sensor_locations_shred, sensor_coords_shred, test_lag_dates_shred = unpack_metadata(f"Reconstructions/Static/Exp1/metadata_n{num_sensors}_l{lags}.json")

In [ ]:
# Check did run SDN model on same test dataset
assert np.unique(test_lag_dates == test_lag_dates_shred)[0]
assert np.unique(ground_truth == ground_truth_shred)[0]


In [ ]:
# Check sensors in same place
plot_sensors(sensor_coords=sensor_coords)
plot_sensors(sensor_coords=sensor_coords_shred)
assert np.unique(sensor_coords == sensor_coords_shred)[0]

## Evaluation

In [ ]:
for i in np.random.choice(range(len(test_recons)), 1):
    recon_df = pd.DataFrame({
        "Latitude": lats,
        "Longitude": longs,
        "SST": test_recons[i]
    })
    shred_df = pd.DataFrame({
        "Latitude": lats,
        "Longitude": longs,
        "SST": test_recons_shred[i]
    })
    true_df = pd.DataFrame({
        "Latitude": lats,
        "Longitude": longs,
        "SST": ground_truth[i]
    })
    print(f"Date: {test_lag_dates[i]}")
    plot_compare_sst_recon(recon_df, true_df, area="NA", sensor_coords=sensor_coords, title=f"SDN reconstruction, Date: {test_lag_dates[i]}")
    plot_compare_sst_recon(shred_df, true_df, area="NA", sensor_coords=sensor_coords_shred, title=f"SHRED reconstruction, Date: {test_lag_dates[i]}")

### Metrics
Structural Similarity Index: [[1]](https://scikit-image.org/docs/stable/api/skimage.metrics.html#skimage.metrics.structural_similarity) [[2]](https://ieeexplore.ieee.org/document/4775883)



#### NA basin

In [ ]:
rmse = np.sqrt(MSE(y_pred=test_recons, y_true=ground_truth))
mae = MAE(y_pred=test_recons, y_true=ground_truth)
r2 = r2_score(y_pred=test_recons, y_true=ground_truth)
print(f"RMSE:\t{rmse:.3f}")
print(f"MAE:\t{mae:.3f}")
print(f"R2:\t{r2:.3f}")

In [ ]:
ssi = np.mean([SSI(ground_truth[i], test_recons[i], 
                   data_range=(max(test_recons[i].max(), ground_truth[i].max()) - min(test_recons[i].min(), ground_truth[i].min())))
                   for i in range(len(test_recons))])
print(f"Average SSI:\t{ssi:.3f}")

#### Gulf-Stream region

In [ ]:
GS_test_data = mask_array_by_lat_long(test_recons, lats, longs)
GS_truth_data = mask_array_by_lat_long(ground_truth, lats, longs)
GS_test_data = [d[~np.isnan(d)] for d in GS_test_data]
GS_truth_data = [d[~np.isnan(d)] for d in GS_truth_data]

In [ ]:
rmse = np.sqrt(MSE(y_pred=GS_test_data, y_true=GS_truth_data))
mae = MAE(y_pred=GS_test_data, y_true=GS_truth_data)
r2 = r2_score(y_pred=GS_test_data, y_true=GS_truth_data)
print(f"RMSE:\t{rmse:.3f}")
print(f"MAE:\t{mae:.3f}")
print(f"R2:\t{r2:.3f}")

In [ ]:
ssi = np.mean([SSI(GS_truth_data[i], GS_test_data[i], data_range=test_recons[i].max() -  test_recons[i].min()) for i in range(len(test_recons))])
print(f"Average SSI:\t{ssi:.3f}")

### Regional integral of SST (area weighted mean)

In [ ]:
test_sst_mean, area_test = area_weighted_mean(test_recons, lats, longs)
truth_sst_mean, area_truth = area_weighted_mean(ground_truth, lats, longs)
test_sst_mean_shred, _ = area_weighted_mean(test_recons_shred, lats, longs)

In [ ]:
sst_mean_df = pd.DataFrame({
    "Date": test_lag_dates,
    "SHRED": test_sst_mean_shred,
    "SDN": test_sst_mean,
    "Truth": truth_sst_mean
})
# Smooth over 1 year = 52 weeks
sst_rolling_df = pd.DataFrame({
    "Date": test_lag_dates,
    "SHRED": sst_mean_df["SHRED"].rolling(52).mean(),
    "SDN": sst_mean_df["SDN"].rolling(52).mean(),
    "Truth": sst_mean_df.Truth.rolling(52).mean()
})
sst_mean_df = sst_mean_df.set_index("Date")
sst_rolling_df = sst_rolling_df.set_index("Date")

In [ ]:
plt.figure(figsize=(8,8))
sst_mean_df[["SHRED", "SDN", "Truth"]].plot(colormap=ListedColormap(colours[:3]), ls='-.')
plt.xticks(rotation=90)
plt.ylabel("Mean SST ($\\degree$C)")
plt.legend(bbox_to_anchor=(1,1))
plt.show()

In [ ]:
plt.figure(figsize=(8,8))
sst_rolling_df[["SHRED", "SDN", "Truth"]].plot(colormap=ListedColormap(colours[:3]), ls='-.')
plt.xticks(rotation=90)
plt.ylabel("Smoothed mean SST ($\\degree$C)")
plt.show()

### Difference between test truth and reconstructions 

In [ ]:
difference = test_recons - ground_truth
difference_shred = test_recons_shred - ground_truth

#### Plots

In [ ]:
plot_index = 3

In [ ]:
for i in [plot_index]:  #range(0, len(test_recons), 10):  # np.random.choice(range(len(test_recons)), 1):
    diff_df = pd.DataFrame({
            "Latitude": lats,
            "Longitude": longs,
            "SST": difference[i]
        })
    recon_df = pd.DataFrame({
        "Latitude": lats,
        "Longitude": longs,
        "SST": test_recons[i]
    })
    true_df = pd.read_csv(f"{NA_DATA_PATH}{test_lag_dates[i]}.csv")
    # plot_sst_map(diff_df, title=f"Difference map for {test_dates[i]}", min_sst=-5, max_sst=5, area="NA", cmap='bwr')
    print(f"Date: {test_lag_dates[i]}")
    plot_compare_sst_recon(recon_df, true_df, diff_data=diff_df, area="NA")  #, sensor_coords=sensor_coords)

    gs_recon_df = crop_df_to_area(recon_df, "GS", "Latitude")
    gs_true_df = crop_df_to_area(true_df, "GS", "Latitude")
    gs_diff_df = crop_df_to_area(diff_df, "GS", "Latitude")
    
    plot_compare_sst_recon(gs_recon_df, gs_true_df, diff_data=gs_diff_df, area='GS')

In [ ]:
plot_sensors(sensor_coords=sensor_coords)

In [ ]:
min_sst = min([min(recon_df.SST), min(true_df.SST)])
max_sst = max([max(recon_df.SST), max(true_df.SST)])
plot_sst_map(recon_df, "Reconstructed SST ($\\degree$C)", min_sst=min_sst, max_sst=max_sst, area="NA")

In [ ]:
plot_sst_map(true_df, "True SST ($\\degree$C)", min_sst=min_sst, max_sst=max_sst, area="NA")

In [ ]:
extent=max(np.abs(min(diff_df.SST)), max(diff_df.SST))
plot_sst_map(diff_df, "Difference ($\\degree$C)", min_sst=-extent, max_sst=extent, area="NA", cmap='bwr')

#### Zonally integrated average difference 
i.e. Average difference between reconstruction and truth across all years;
plotted against latitude (no weighting by area needed, because each square in latitude band has same area)

OR could scale by area by doing lat_band/total_area????

In [ ]:
latitude_avg_diffs = np.array([pd.Series(diff).groupby(lats).mean().to_numpy() for diff in difference])
avg_all_time_lat_avg_diffs = np.array([np.mean(latitude_avg_diffs.T[i]) for i in range(latitude_avg_diffs.shape[1])])

latitude_avg_diffs_shred = np.array([pd.Series(diff).groupby(lats).mean().to_numpy() for diff in difference_shred])
avg_all_time_lat_avg_diffs_shred = np.array([np.mean(latitude_avg_diffs_shred.T[i]) for i in range(latitude_avg_diffs_shred.shape[1])])

In [ ]:
plt.plot(avg_all_time_lat_avg_diffs_shred, np.unique(lats), c=colours[0], label="SHRED")
plt.plot(avg_all_time_lat_avg_diffs, np.unique(lats), c=colours[1], label="SDN")
plt.vlines(0, 0, 60, colors=['k'], linestyles=['--'])
plt.xlabel('Average difference (from the truth) ($\\degree$C)')
plt.ylabel('Latitude')
plt.legend()

NOTE: not even numbers of data points in each year

In [ ]:
years = [int(date.year) for date in test_lag_dates]
year_labels = np.unique(years)
Counter(years)

In [ ]:
for title, lat_diff in {"SHRED": latitude_avg_diffs_shred, "SDN": latitude_avg_diffs}.items():
    diff_by_years = np.array([[time[j] for time in lat_diff] for j in range(len(lat_diff[0]))])
    latitude_avg_diffs_avg_by_year = np.array([pd.Series(d).groupby(years).mean().to_numpy() for d in diff_by_years])

    for i in range(len(year_labels)):
        lat_for_year = [lat[i] for lat in latitude_avg_diffs_avg_by_year]
        plt.plot(lat_for_year, np.unique(lats), label=year_labels[i], c=colours[i])
    plt.vlines(0, 0, 60, colors=['k'], linestyles=['--'])
    plt.xlabel('Average difference (from the truth) ($\\degree$C)')
    plt.ylabel('Latitude')
    plt.title(title)
    plt.legend()
    plt.show()

#### Average difference between reconstruction and truth across whole NA region - by year

In [ ]:
avg_difference = [np.mean(d) for d in difference]
avg_all_time_difference = np.mean(avg_difference)
print(avg_all_time_difference)
plt.figure(figsize=(8,6))
plt.stem(test_lag_dates, avg_difference, markerfmt='.')
plt.ylabel('Average difference ($\\degree$C)')
plt.xlabel('Reconstruction date')

In [ ]:
# For Gulf Stream only:
GS_data = mask_array_by_lat_long(difference, lats, longs)
GS_data = [d[~np.isnan(d)] for d in GS_data]

In [ ]:
avg_difference = [np.mean(d) for d in GS_data]
avg_all_time_difference = np.mean(avg_difference)
print(avg_all_time_difference)
plt.figure(figsize=(8,6))
plt.stem(test_lag_dates, avg_difference, markerfmt='.')
plt.ylabel('Average difference ($\\degree$C)')
plt.xlabel('Reconstruction date')